In [26]:
#installing needed libraries and models
%pip install insightface
%pip install onnxruntime-gpu
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.
ERROR: Could not find a version that satisfies the requirement onnxruntime-gpu (from versions: none)
ERROR: No matching distribution found for onnxruntime-gpu
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [27]:
# importing libraries
import os
import cv2
import numpy as np
from tqdm import tqdm
from insightface.app import FaceAnalysis

In [28]:
# loading arcface
print("Loading ArcFace...")

app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0, det_size=(640, 640))

rec_model = app.models["recognition"]

print("ArcFace ready!")

Loading ArcFace...
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0


/opt/miniconda3/lib/python3.13/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:149: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'CoreMLExecutionProvider, AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)
ArcFace ready!


In [29]:
# extracting the embeddings using arcface
def save_embeddings_preserve_structure(
    image_root,
    embedding_root
):
    total = 0

    for root, dirs, files in os.walk(image_root):

        rel_path = os.path.relpath(root, image_root)

        save_dir = os.path.join(
            embedding_root,
            rel_path
        )

        os.makedirs(save_dir, exist_ok=True)

        for file in files:

            if not file.lower().endswith(
                (".jpg", ".jpeg", ".png", ".bmp")
            ):
                continue

            img_path = os.path.join(root, file)

            img = cv2.imread(img_path)

            if img is None:
                print("Failed:", img_path)
                continue

            img = cv2.resize(img, (112, 112))
            img = cv2.cvtColor(
                img,
                cv2.COLOR_BGR2RGB
            )

            embedding = (
                rec_model
                .get_feat(img)
                .flatten()
                .astype(np.float32)
            )

            save_path = os.path.join(
                save_dir,
                os.path.splitext(file)[0] + ".npy"
            )

            np.save(save_path, embedding)

            total += 1

    print(f"Saved {total} embeddings")

In [30]:
save_embeddings_preserve_structure(
    "dataset_split/train/gallery_cropped",
    "embeddings/train/gallery"
)

save_embeddings_preserve_structure(
    "dataset_split/test/gallery_cropped",
    "embeddings/test/gallery"
)

Saved 2002 embeddings
Saved 482 embeddings


In [31]:
save_embeddings_preserve_structure(
    "dataset_split/train/degraded_probes",
    "embeddings/train/degraded_probes"
)
save_embeddings_preserve_structure(
    "dataset_split/test/degraded_probes",
    "embeddings/test/degraded_probes"
)

Saved 2196 embeddings
Saved 540 embeddings


In [35]:
import numpy as np

sample = np.load(
    "/Users/admin/Desktop/reliable_rejection_under_degradation/second_half/embeddings/train/degraded_probes/Abdullah_Gul/Abdullah_Gul_0002_blur_high.npy"
)

print(sample.shape)
print(sample.dtype)

(512,)
float32


In [33]:
# backup zip
import shutil

zip_path = shutil.make_archive(
    "arcface_embeddings_backup",
    "zip",
    "embeddings"
)

print("Backup created:")
print(zip_path)

Backup created:
/Users/admin/Desktop/reliable_rejection_under_degradation/second_half/arcface_embeddings_backup.zip
